In this notebook, we will build a 3D map of a scene from a small set of images and then localize an image downloaded from the Internet. This demo was contributed by [Philipp Lindenberger](https://github.com/Phil26AT/).

In [1]:
from pathlib import Path

import csv
import pickle

import h5py
import numpy as np
import pycolmap
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation

from hloc import (
    extract_features,
    match_features,
    pairs_from_retrieval,
    localize_sfm,
)

# Setup
Here we define some output paths.

In [2]:
mapping_dir = Path(
    "datasets/my_desk/mapping"
)

query_dir = Path(
    "datasets/my_desk/query"
)

outputs = Path(
    "outputs/my_desk_v2"
)

sfm_dir = outputs / "sfm"

mapping_local_features = (
    outputs / "features.h5"
)

query_local_features = (
    outputs / "features_query.h5"
)

mapping_global_features = (
    outputs / "global_mapping_netvlad.h5"
)

query_global_features = (
    outputs / "global_query_netvlad.h5"
)

loc_pairs = outputs / "pairs-query.txt"

loc_matches = (
    outputs
    / "matches-query-aliked-lightglue.h5"
)

queries_file = (
    outputs / "queries_with_intrinsics.txt"
)

pose_results = outputs / "query_poses.txt"

print("地图模型：", sfm_dir.resolve())
print("查询图片：", query_dir.resolve())

# 3D mapping
First we list the images used for mapping. These are all day-time shots of Sacre Coeur.

In [3]:
model = pycolmap.Reconstruction(
    str(sfm_dir)
)

image_suffixes = {
    ".jpg",
    ".jpeg",
    ".png",
    ".JPG",
    ".JPEG",
    ".PNG",
}

mapping_list = sorted(
    p.relative_to(mapping_dir).as_posix()
    for p in mapping_dir.rglob("*")
    if p.is_file() and p.suffix in image_suffixes
)

query_list = sorted(
    p.relative_to(query_dir).as_posix()
    for p in query_dir.rglob("*")
    if p.is_file() and p.suffix in image_suffixes
)

print(model)
print("建图图片数量：", len(mapping_list))
print("查询图片数量：", len(query_list))
print("查询图片：", query_list)

312 mapping images


Then we extract features and match them across image pairs. Since we deal with few images, we simply match all pairs exhaustively. For larger scenes, we would use image retrieval, as demonstrated in the other notebooks.

In [4]:
local_conf = extract_features.confs[
    "aliked-n16"
]

extract_features.main(
    local_conf,
    query_dir,
    image_list=query_list,
    feature_path=query_local_features,
    overwrite=False,
)

print(
    "查询局部特征：",
    query_local_features.resolve(),
)

[2026/07/20 14:29:32 hloc INFO] Extracting local features with configuration:
{'model': {'model_name': 'aliked-n16', 'name': 'aliked'},
 'output': 'feats-aliked-n16',
 'preprocessing': {'grayscale': False, 'resize_max': 1024}}


  0%|          | 0/312 [00:00<?, ?it/s]

[2026/07/20 14:29:56 hloc INFO] Finished exporting features.
[2026/07/20 14:29:56 hloc INFO] Found 48516 pairs.
[2026/07/20 14:29:56 hloc INFO] Matching local features with configuration:
{'model': {'features': 'aliked', 'name': 'lightglue'},
 'output': 'matches-aliked-lightglue'}


  0%|          | 0/48516 [00:00<?, ?it/s]

[2026/07/20 15:14:44 hloc INFO] Finished exporting matches.


WindowsPath('outputs/my_desk_v2/matches.h5')

The we run incremental Structure-From-Motion and display the reconstructed 3D model.

In [5]:
global_conf = extract_features.confs[
    "netvlad"
]

extract_features.main(
    global_conf,
    mapping_dir,
    image_list=mapping_list,
    feature_path=mapping_global_features,
    overwrite=False,
)

extract_features.main(
    global_conf,
    query_dir,
    image_list=query_list,
    feature_path=query_global_features,
    overwrite=False,
)

print(
    "地图全局特征：",
    mapping_global_features.resolve(),
)

print(
    "查询全局特征：",
    query_global_features.resolve(),
)

[2026/07/20 15:15:39 hloc INFO] Writing COLMAP logs to outputs\my_desk_v2\sfm\colmap.LOG.*
[2026/07/20 15:15:39 hloc INFO] Creating an empty database...
[2026/07/20 15:15:39 hloc INFO] Importing images into the database...
[2026/07/20 15:15:42 hloc INFO] Importing features into the database...


  0%|          | 0/312 [00:00<?, ?it/s]

[2026/07/20 15:15:42 hloc INFO] Importing matches into the database...


  0%|          | 0/48516 [00:00<?, ?it/s]

[2026/07/20 15:16:49 hloc INFO] Performing geometric verification of the matches...
[2026/07/20 15:17:11 hloc INFO] Running 3D reconstruction...


Reconstruction 0:   0%|          | 0/312 [00:00<?, ?images/s, registered]

[2026/07/20 15:30:53 hloc INFO] Reconstructed 1 model(s).
[2026/07/20 15:30:53 hloc INFO] Largest model is #0 with 304 images.
[2026/07/20 15:30:53 hloc INFO] Reconstruction statistics:
Reconstruction:
	num_rigs = 304
	num_cameras = 304
	num_frames = 304
	num_reg_frames = 304
	num_images = 304
	num_points3D = 9842
	num_observations = 133277
	mean_track_length = 13.5417
	mean_observations_per_image = 438.411
	mean_reprojection_error = 1.41807
	num_input_images = 312


Reconstruction(num_rigs=304, num_cameras=304, num_frames=304, num_reg_frames=304, num_images=304, num_points3D=9842)


In [1]:
pairs_from_retrieval.main(
    descriptors=query_global_features,
    output=loc_pairs,
    num_matched=20,
    db_model=sfm_dir,
    db_descriptors=mapping_global_features,
)

pair_lines = loc_pairs.read_text(
    encoding="utf-8"
).splitlines()

print("检索对数量：", len(pair_lines))

for line in pair_lines[:20]:
    print(line)

NameError: name 'viz_3d' is not defined

We also visualize which keypoints were triangulated into the 3D model.

In [ ]:
matcher_conf = match_features.confs[
    "aliked+lightglue"
]

match_features.main(
    matcher_conf,
    loc_pairs,
    features=query_local_features,
    features_ref=mapping_local_features,
    matches=loc_matches,
    overwrite=False,
)

print("匹配结果：", loc_matches.resolve())

from pathlib import Path

from hloc.utils import viz_3d
from hloc import reconstruction


In [3]:
query_paths = [
    query_dir / name
    for name in query_list
]

with queries_file.open(
    "w",
    encoding="utf-8",
) as file:

    for image_path in query_paths:
        camera = pycolmap.infer_camera_from_image(
            str(image_path)
        )

        params = " ".join(
            format(float(value), ".12g")
            for value in camera.params
        )

        image_name = (
            image_path
            .relative_to(query_dir)
            .as_posix()
        )

        file.write(
            f"{image_name} "
            f"{camera.model_name} "
            f"{camera.width} "
            f"{camera.height} "
            f"{params}\n"
        )

print(
    queries_file.read_text(
        encoding="utf-8"
    )
)

NameError: name 'Path' is not defined

# Localization
Now that we have a 3D map of the scene, we can localize any image. To demonstrate this, we download [a night-time image from Wikimedia](https://commons.wikimedia.org/wiki/File:Paris_-_Basilique_du_Sacr%C3%A9_Coeur,_Montmartre_-_panoramio.jpg).

In [ ]:
localize_sfm.main(
    reference_sfm=sfm_dir,
    queries=queries_file,
    retrieval=loc_pairs,
    features=query_local_features,
    matches=loc_matches,
    results=pose_results,
    ransac_thresh=12,
    covisibility_clustering=False,
)

print("位姿文件：", pose_results.resolve())

Again, we extract features for the query and match them exhaustively.

In [ ]:
logs_file = Path(
    str(pose_results) + "_logs.pkl"
)

with logs_file.open("rb") as file:
    logs = pickle.load(file)

quality = {}

for image_name, info in logs["loc"].items():
    result = info.get("PnP_ret")

    if result is None:
        quality[image_name] = {
            "pnp_success": False,
            "num_inliers": 0,
            "num_correspondences": 0,
            "inlier_ratio": 0.0,
        }
        continue

    inlier_mask = result.get("inlier_mask")

    total = (
        len(inlier_mask)
        if inlier_mask is not None
        else 0
    )

    num_inliers = int(
        result.get("num_inliers", 0)
    )

    quality[image_name] = {
        "pnp_success": True,
        "num_inliers": num_inliers,
        "num_correspondences": total,
        "inlier_ratio": (
            num_inliers / total
            if total else 0.0
        ),
    }


poses = {}

for line in pose_results.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    values = line.split()

    image_name = values[0]

    qw, qx, qy, qz = map(
        float,
        values[1:5],
    )

    tx, ty, tz = map(
        float,
        values[5:8],
    )

    rotation = Rotation.from_quat(
        [qx, qy, qz, qw]
    ).as_matrix()

    translation = np.array(
        [tx, ty, tz]
    )

    camera_center = (
        -rotation.T @ translation
    )

    poses[image_name] = {
        "qw": qw,
        "qx": qx,
        "qy": qy,
        "qz": qz,
        "tx": tx,
        "ty": ty,
        "tz": tz,
        "X": float(camera_center[0]),
        "Y": float(camera_center[1]),
        "Z": float(camera_center[2]),
    }


for image_name in sorted(poses):
    pose = poses[image_name]
    result = quality[image_name]

    print(
        f"{image_name}: "
        f"X={pose['X']:.6f}, "
        f"Y={pose['Y']:.6f}, "
        f"Z={pose['Z']:.6f}, "
        f"内点={result['num_inliers']}/"
        f"{result['num_correspondences']}"
    )

We read the EXIF data of the query to infer a rough initial estimate of camera parameters like the focal length. Then we estimate the absolute camera pose using PnP+RANSAC and refine the camera parameters.

In [ ]:
csv_path = (
    outputs / "localization_results.csv"
)

fieldnames = [
    "image",
    "pnp_success",
    "num_inliers",
    "num_correspondences",
    "inlier_ratio",
    "qw",
    "qx",
    "qy",
    "qz",
    "tx",
    "ty",
    "tz",
    "X",
    "Y",
    "Z",
]

rows = []

for image_name in sorted(poses):
    row = {
        "image": image_name,
        **quality[image_name],
        **poses[image_name],
    }

    rows.append(row)

with csv_path.open(
    "w",
    newline="",
    encoding="utf-8-sig",
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=fieldnames,
    )

    writer.writeheader()
    writer.writerows(rows)

print("CSV结果：", csv_path.resolve())

We visualize the correspondences between the query images a few mapping images. We can also visualize the estimated camera pose in the 3D map.

In [ ]:
map_points = np.array(
    [
        point.xyz
        for point in model.points3D.values()
    ]
)

max_points = 6000

if len(map_points) > max_points:
    indices = np.linspace(
        0,
        len(map_points) - 1,
        max_points,
        dtype=int,
    )

    map_points = map_points[indices]


query_names = sorted(poses)

query_xyz = np.array(
    [
        [
            poses[name]["X"],
            poses[name]["Y"],
            poses[name]["Z"],
        ]
        for name in query_names
    ]
)


figure = go.Figure()


 ​:contentReference[oaicite:1]{index=1}​

from pathlib import Path

sfm = Path("outputs/demo/sfm")

for f in sfm.iterdir():
    print(f)

outputs\demo\sfm\cameras.bin
outputs\demo\sfm\colmap.LOG.20260716-161255.31528
outputs\demo\sfm\database.db
outputs\demo\sfm\frames.bin
outputs\demo\sfm\images.bin
outputs\demo\sfm\models
outputs\demo\sfm\points3D.bin
outputs\demo\sfm\rigs.bin


ValueError: [reconstruction.cc:983] rigs, cameras, frames, images, points3D files do not exist at "outputs\\my_desk\\sfm"

当前工作目录： E:\Hierarchical-Localization-master
输出目录是否存在： False
输出目录绝对路径： E:\Hierarchical-Localization-master\outputs\my_desk
共找到模型文件： 0


model 是否还在内存： False


正在搜索 E 盘，可能需要几分钟……
找到完整模型： E:\Hierarchical-Localization-master\outputs\demo\sfm

搜索完成。
完整模型目录数量： 1


Reconstruction(num_rigs=10, num_cameras=10, num_frames=10, num_reg_frames=10, num_images=10, num_points3D=2814)


Reconstruction(num_rigs=304, num_cameras=304, num_frames=304, num_reg_frames=304, num_images=304, num_points3D=9842)


[WindowsPath('outputs/my_desk_v2/sfm/cameras.bin'), WindowsPath('outputs/my_desk_v2/sfm/colmap.LOG.20260720-151649.36700'), WindowsPath('outputs/my_desk_v2/sfm/database.db'), WindowsPath('outputs/my_desk_v2/sfm/frames.bin'), WindowsPath('outputs/my_desk_v2/sfm/images.bin'), WindowsPath('outputs/my_desk_v2/sfm/models'), WindowsPath('outputs/my_desk_v2/sfm/points3D.bin'), WindowsPath('outputs/my_desk_v2/sfm/rigs.bin')]
